# AI Speak — training

Trains a model that predicts **52 ARKit blendshape coefficients** (60 FPS) from speech audio.

Pipeline
1. Audio → **HuBERT** (`facebook/hubert-base-ls960`) features, 768-dim at ~50 FPS
2. Features are interpolated to the 60 FPS blendshape timeline
3. A **BiLSTM** looks at an 11-frame window (5 before + current + 5 after) and predicts the blendshapes of the centre frame

The output of this notebook is `best_model_v1.pth`, which `inference.ipynb` uses.

> Runtime → Change runtime type → **GPU** before running.

## 1. Setup

In [ ]:
!pip install transformers librosa soundfile tqdm -q

import os
import time
import numpy as np
import librosa
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel, Wav2Vec2FeatureExtractor
from scipy.interpolate import interp1d
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 2. Paths

Expected data layout (one folder per speaker, each `.csv` has a matching `.wav`)
```
BASE/
├── spk08_blendshapes/renamed_spk08/  spk08_001.wav, spk08_001.csv, ...
└── spk14_blendshapes/renamed_spk14/  spk14_001.wav, spk14_001.csv, ...
```
Each CSV has one row per frame (60 FPS) and 52 columns (ARKit blendshapes, values 0–1, no header).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 👉 Change this to the folder where your data is
BASE = '/content/drive/MyDrive/Colab Notebooks/AISpeak_Libraries'

SPEAKERS = ['spk08', 'spk14']
MODEL_OUT = os.path.join(BASE, 'best_model_v1.pth')

## 3. HuBERT feature extractor

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
hubert.eval()
hubert.to(DEVICE)
print("HuBERT loaded.")


def align_features_to_blendshapes(hubert_features, num_blendshape_frames):
    """Linearly interpolate HuBERT features (~50 FPS) to the blendshape timeline (60 FPS).

    hubert_features: (hubert_frames, 768)
    returns:         (num_blendshape_frames, 768)
    """
    hubert_times = np.linspace(0, 1, len(hubert_features))
    target_times = np.linspace(0, 1, num_blendshape_frames)
    interpolator = interp1d(hubert_times, hubert_features, axis=0, kind='linear')
    return interpolator(target_times).astype(np.float32)

## 4. Extract features for all speakers

In [ ]:
def process_all_data(base_path, speakers):
    """For every WAV+CSV pair, extract HuBERT features and align them to the blendshapes."""
    all_features, all_targets, skipped = [], [], []

    for spk in speakers:
        folder = os.path.join(base_path, f'{spk}_blendshapes/renamed_{spk}')
        csv_files = sorted(f for f in os.listdir(folder) if f.endswith('.csv'))
        print(f"\nProcessing {spk}: {len(csv_files)} files")

        for csv_file in tqdm(csv_files):
            csv_path = os.path.join(folder, csv_file)
            wav_path = os.path.join(folder, csv_file.replace('.csv', '.wav'))

            if not os.path.exists(wav_path):
                skipped.append(csv_file)
                continue

            try:
                bs = np.loadtxt(csv_path, delimiter=',', dtype=np.float32)
                audio, _ = librosa.load(wav_path, sr=16000)  # HuBERT expects 16 kHz

                with torch.no_grad():
                    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
                    feats = hubert(inputs.input_values.to(DEVICE)).last_hidden_state
                    feats = feats.squeeze(0).cpu().numpy()

                all_features.append(align_features_to_blendshapes(feats, len(bs)))
                all_targets.append(bs)
            except Exception as e:
                skipped.append(f"{csv_file}: {e}")

    if skipped:
        print(f"\nSkipped {len(skipped)} files, e.g. {skipped[:5]}")
    return all_features, all_targets


all_features, all_targets = process_all_data(BASE, SPEAKERS)

total_frames = sum(len(f) for f in all_features)
print(f"\nFiles: {len(all_features)} | frames: {total_frames} | duration: {total_frames/60/60:.1f} min")

## 5. Dataset and train/val split

Each training sample is an 11-frame window of features (≈183 ms) → the 52 blendshapes of the centre frame.
The split is done **by file**, so frames from the same recording never end up in both train and val.

In [ ]:
CONTEXT = 5  # 5 frames before + current + 5 after = 11 frames


class BlendshapeDataset(Dataset):
    def __init__(self, features_list, targets_list, context_frames=CONTEXT):
        self.data = []
        for feats, targs in zip(features_list, targets_list):
            pad = np.zeros((context_frames, feats.shape[1]), dtype=np.float32)
            padded = np.concatenate([pad, feats, pad], axis=0)
            for i in range(len(targs)):
                window = padded[i : i + 2 * context_frames + 1]  # (11, 768)
                self.data.append((window, targs[i]))             # target (52,)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        window, target = self.data[idx]
        return torch.tensor(window, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)


np.random.seed(42)
indices = np.random.permutation(len(all_features))
split = int(0.9 * len(all_features))
train_idx, val_idx = indices[:split], indices[split:]

train_features = [all_features[i] for i in train_idx]
train_targets  = [all_targets[i]  for i in train_idx]
val_features   = [all_features[i] for i in val_idx]
val_targets    = [all_targets[i]  for i in val_idx]

train_dataset = BlendshapeDataset(train_features, train_targets)
val_dataset   = BlendshapeDataset(val_features, val_targets)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} frames ({len(train_idx)} files)")
print(f"Val:   {len(val_dataset)} frames ({len(val_idx)} files)")

## 6. Model

`Linear 768→256` → `2-layer BiLSTM (256)` → take the centre frame → `Linear 512→256 → ReLU → Dropout → Linear 256→52 → Sigmoid`

≈ 2.97M parameters.

In [ ]:
class BlendshapeModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, output_dim=52):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )
        self.output_layers = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),  # blendshape values are in [0, 1]
        )

    def forward(self, x):                                  # (batch, 11, 768)
        x = self.input_proj(x)                             # (batch, 11, 256)
        lstm_out, _ = self.lstm(x)                         # (batch, 11, 512)
        center = lstm_out[:, lstm_out.shape[1] // 2, :]    # (batch, 512)
        return self.output_layers(center)                  # (batch, 52)


model = BlendshapeModel().to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Training

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

NUM_EPOCHS = 30
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(NUM_EPOCHS):
    start = time.time()

    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item()
    train_loss = total / len(train_loader)

    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            total += criterion(model(x), y).item()
    val_loss = total / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_OUT)
        marker = "  <-- best"

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | train {train_loss:.6f} | val {val_loss:.6f} | {time.time()-start:.1f}s{marker}")

print(f"\nBest validation loss: {best_val_loss:.6f}")
print(f"Model saved to {MODEL_OUT}")

## 8. Results

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_title('MSE loss')
axes[1].plot(range(2, len(train_losses)), train_losses[2:], label='Train')
axes[1].plot(range(2, len(val_losses)), val_losses[2:], label='Val')
axes[1].set_title('MSE loss (from epoch 3, zoomed)')
for ax in axes:
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

# Ground truth vs prediction on one validation file
model.load_state_dict(torch.load(MODEL_OUT, map_location=DEVICE))
model.eval()

test_feat, test_targ = val_features[0], val_targets[0]
pad = np.zeros((CONTEXT, test_feat.shape[1]), dtype=np.float32)
padded = np.concatenate([pad, test_feat, pad], axis=0)
windows = np.stack([padded[i : i + 2 * CONTEXT + 1] for i in range(len(test_targ))])
with torch.no_grad():
    predictions = model(torch.tensor(windows).to(DEVICE)).cpu().numpy()

mouth = {24: 'jawOpen', 28: 'mouthFunnel', 29: 'mouthPucker', 36: 'mouthClose',
         37: 'mouthSmileLeft', 38: 'mouthSmileRight', 45: 'mouthLowerDownLeft', 46: 'mouthLowerDownRight'}

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
for ax, (idx, name) in zip(axes.flatten(), mouth.items()):
    ax.plot(test_targ[:, idx], label='Ground truth', alpha=0.7)
    ax.plot(predictions[:, idx], label='Predicted', alpha=0.7)
    ax.set_title(name); ax.legend(); ax.grid(True)
plt.suptitle('Ground truth vs predicted blendshapes (validation file)', fontsize=14)
plt.tight_layout(); plt.show()